In [ ]:
pip install requests pandas

In [ ]:
import requests
import pandas as pd
import time
from pathlib import Path

In [ ]:
# konfigurasi
BASE_URL = "https://api.gbif.org/v1/occurrence/search"
SOUTHEAST_ASIA_COUNTRIES = {
    "ID": "Indonesia",
    "MY": "Malaysia",
    "SG": "Singapore",
    "TH": "Thailand",
    "VN": "Vietnam",
    "PH": "Philippines"
}

YEARS = [2020, 2021, 2022, 2023, 2024, 2025, 2026]
RECORDS_PER_COUNTRY_PER_YEAR = 120
PAGE_SIZE = 100
OUTPUT_DIR = Path("data")
OUTPUT_DIR.mkdir(exist_ok=True)
OUTPUT_FILE = OUTPUT_DIR / "gbif_biodiversity_graph.csv"
SPECIES_TEXT_FILE = OUTPUT_DIR / "species_descriptions.csv"
headers = {
    "User-Agent": "Neo4jBiodiversityGraph/1.0"
}

In [ ]:
def fetch_gbif_occurrences(total_records=120, page_size=100, country_code="ID", year=2026):
    all_rows = []
    for offset in range(0, total_records, page_size):
        current_limit = min(page_size, total_records - offset)
        params = {
            "country": country_code,
            "year": year,
            "hasCoordinate": "true",
            "hasGeospatialIssue": "false",
            "limit": current_limit,
            "offset": offset}

        print(f"Fetching {country_code} year {year} offset {offset} ...")
        response = requests.get(BASE_URL, params=params, headers=headers, timeout=30)

        response.raise_for_status()
        data = response.json()
        records = data.get("results", [])

        if not records:
            print(f"Tidak ada data tambahan untuk {country_code} tahun {year}.")
            break

        for r in records:
            row = {
                "occurrenceKey": r.get("key") or r.get("gbifID"),
                "taxonKey": r.get("taxonKey"),
                "scientificName": r.get("scientificName"),
                "kingdomKey": r.get("kingdomKey"),
                "kingdom": r.get("kingdom"),
                "phylumKey": r.get("phylumKey"),
                "phylum": r.get("phylum"),
                "classKey": r.get("classKey"),
                "className": r.get("class"),
                "orderKey": r.get("orderKey"),
                "orderName": r.get("order"),
                "familyKey": r.get("familyKey"),
                "family": r.get("family"),
                "genusKey": r.get("genusKey"),
                "genus": r.get("genus"),
                "speciesKey": r.get("speciesKey"),
                "species": r.get("species"),
                "countryCode": r.get("countryCode"),
                "country": r.get("country"),
                "decimalLatitude": r.get("decimalLatitude"),
                "decimalLongitude": r.get("decimalLongitude"),
                "year": r.get("year"),
                "basisOfRecord": r.get("basisOfRecord")
            }

            all_rows.append(row)
        time.sleep(0.3)
    return pd.DataFrame(all_rows)

In [ ]:
# Ambil data dari seluruh negara Asia Tenggara yang dipilih
all_dfs = []
for code, country_name in SOUTHEAST_ASIA_COUNTRIES.items():
    for year in YEARS:
        print(f"\nMengambil data untuk {country_name} ({code}) tahun {year}")
        df_country_year = fetch_gbif_occurrences(
            total_records=RECORDS_PER_COUNTRY_PER_YEAR,
            page_size=PAGE_SIZE,
            country_code=code,
            year=year)

        all_dfs.append(df_country_year)
df_raw = pd.concat(all_dfs, ignore_index=True)
print("Ukuran data mentah:", df_raw.shape)
display(df_raw.head())


Mengambil data untuk Indonesia (ID) tahun 2020
Fetching ID year 2020 offset 0 ...
Fetching ID year 2020 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2021
Fetching ID year 2021 offset 0 ...
Fetching ID year 2021 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2022
Fetching ID year 2022 offset 0 ...
Fetching ID year 2022 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2023
Fetching ID year 2023 offset 0 ...
Fetching ID year 2023 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2024
Fetching ID year 2024 offset 0 ...
Fetching ID year 2024 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2025
Fetching ID year 2025 offset 0 ...
Fetching ID year 2025 offset 100 ...

Mengambil data untuk Indonesia (ID) tahun 2026
Fetching ID year 2026 offset 0 ...
Fetching ID year 2026 offset 100 ...

Mengambil data untuk Malaysia (MY) tahun 2020
Fetching MY year 2020 offset 0 ...
Fetching MY year 2020 offset 100 ...

Mengambil data untuk Malaysia (MY) tahun

,occurrenceKey,taxonKey,scientificName,kingdomKey,kingdom,phylumKey,phylum,classKey,className,orderKey,...,genusKey,genus,speciesKey,species,countryCode,country,decimalLatitude,decimalLongitude,year,basisOfRecord
0,2542914614,2158332.0,"Poltys illepidus C.L.Koch, 1843",1.0,Animalia,54.0,Arthropoda,367.0,Arachnida,1496.0,...,2158311.0,Poltys,2158332.0,Poltys illepidus,ID,Indonesia,-6.347636,106.759209,2020,HUMAN_OBSERVATION
1,2542919124,3034128.0,Centella asiatica (L.) Urb.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1351.0,...,3034124.0,Centella,3034128.0,Centella asiatica,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION
2,2542919768,3068937.0,Euphorbia hirta L.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1414.0,...,11397237.0,Euphorbia,3068937.0,Euphorbia hirta,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION
3,2543078675,9125207.0,"Calotes versicolor (Daudin, 1802)",1.0,Animalia,44.0,Chordata,11592253.0,Squamata,NaN,...,2466496.0,Calotes,9125207.0,Calotes versicolor,ID,Indonesia,-6.518444,107.392075,2020,HUMAN_OBSERVATION
4,2543080039,4956626.0,"Sulcospira testudinaria Busch, 1842",1.0,Animalia,52.0,Mollusca,225.0,Gastropoda,NaN,...,4608242.0,Sulcospira,4956626.0,Sulcospira testudinaria,ID,Indonesia,-7.722783,110.352478,2020,HUMAN_OBSERVATION


In [ ]:
# cleaning data
# Kolom wajib agar struktur graph lengkap dan gk banyak node kosong
required_cols = [
    "occurrenceKey",
    "taxonKey",
    "scientificName",
    "kingdomKey", "kingdom",
    "phylumKey", "phylum",
    "classKey", "className",
    "orderKey", "orderName",
    "familyKey", "family",
    "genusKey", "genus",
    "speciesKey", "species",
    "countryCode", "country",
    "decimalLatitude",
    "decimalLongitude",
    "year",
    "basisOfRecord"
]

df_clean = df_raw.copy()
for col in df_clean.columns:
    if df_clean[col].dtype == "object":
        df_clean[col] = df_clean[col].astype(str).str.strip()
        df_clean[col] = df_clean[col].replace({
            "": pd.NA,
            "None": pd.NA,
            "nan": pd.NA})
df_clean = df_clean.dropna(subset=required_cols)
df_clean["decimalLatitude"] = pd.to_numeric(
    df_clean["decimalLatitude"],
    errors="coerce")
df_clean["decimalLongitude"] = pd.to_numeric(
    df_clean["decimalLongitude"],
    errors="coerce")
df_clean["year"] = pd.to_numeric(
    df_clean["year"],
    errors="coerce").astype("Int64")

df_clean = df_clean.dropna(
    subset=["decimalLatitude", "decimalLongitude", "year"])
df_clean = df_clean.drop_duplicates(subset=["occurrenceKey"])
df_clean = df_clean.reset_index(drop=True)
df_clean["region"] = "Southeast Asia"
df_clean["source"] = "GBIF Occurrence API"

print("Ukuran data bersih:", df_clean.shape)
display(df_clean.head())

Ukuran data bersih: (4264, 25)


,occurrenceKey,taxonKey,scientificName,kingdomKey,kingdom,phylumKey,phylum,classKey,className,orderKey,...,speciesKey,species,countryCode,country,decimalLatitude,decimalLongitude,year,basisOfRecord,region,source
0,2542914614,2158332.0,"Poltys illepidus C.L.Koch, 1843",1.0,Animalia,54.0,Arthropoda,367.0,Arachnida,1496.0,...,2158332.0,Poltys illepidus,ID,Indonesia,-6.347636,106.759209,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
1,2542919124,3034128.0,Centella asiatica (L.) Urb.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1351.0,...,3034128.0,Centella asiatica,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
2,2542919768,3068937.0,Euphorbia hirta L.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1414.0,...,3068937.0,Euphorbia hirta,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
3,2543080789,2791514.0,Bryobium retusum (Blume) Y.P.Ng & P.J.Cribb,6.0,Plantae,7707728.0,Tracheophyta,196.0,Liliopsida,1169.0,...,2791514.0,Bryobium retusum,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
4,2543081090,5307066.0,Crepidium koordersii (J.J.Sm.) Szlach.,6.0,Plantae,7707728.0,Tracheophyta,196.0,Liliopsida,1169.0,...,5307066.0,Crepidium koordersii,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API


In [ ]:
# cek null
null_report = (df_clean.isnull().mean().mul(100).round(2).sort_values())
print("Persentase null setelah cleaning:")
print(null_report)

Persentase null setelah cleaning:
occurrenceKey       0.0
taxonKey            0.0
scientificName      0.0
kingdomKey          0.0
kingdom             0.0
phylumKey           0.0
phylum              0.0
classKey            0.0
className           0.0
orderKey            0.0
orderName           0.0
familyKey           0.0
family              0.0
genusKey            0.0
genus               0.0
speciesKey          0.0
species             0.0
countryCode         0.0
country             0.0
decimalLatitude     0.0
decimalLongitude    0.0
year                0.0
basisOfRecord       0.0
region              0.0
source              0.0
dtype: float64


In [ ]:
entity_summary = {
    "Occurrence": df_clean["occurrenceKey"].nunique(),
    "Taxon": df_clean["taxonKey"].nunique(),
    "Scientific Name": df_clean["scientificName"].nunique(),
    "Species": df_clean["species"].nunique(),
    "Genus": df_clean["genus"].nunique(),
    "Family": df_clean["family"].nunique(),
    "Order": df_clean["orderName"].nunique(),
    "Class": df_clean["className"].nunique(),
    "Phylum": df_clean["phylum"].nunique(),
    "Kingdom": df_clean["kingdom"].nunique(),
    "Country": df_clean["country"].nunique(),
    "Year": df_clean["year"].nunique(),
    "BasisOfRecord": df_clean["basisOfRecord"].nunique(),
    "Region": df_clean["region"].nunique()
}
entity_summary_df = pd.DataFrame(
    list(entity_summary.items()),
    columns=["Entity", "Total Unique"]
)
display(entity_summary_df)

,Entity,Total Unique
0,Occurrence,4264
1,Taxon,1608
2,Scientific Name,1608
3,Species,1542
4,Genus,1086
5,Family,426
6,Order,142
7,Class,31
8,Phylum,11
9,Kingdom,3


In [ ]:
print("Jumlah data bersih per negara:")
display(df_clean["country"].value_counts())
print("Jumlah data berdasarkan tahun:")
display(df_clean["year"].value_counts().sort_index())
print("Jumlah data berdasarkan basisOfRecord:")
display(df_clean["basisOfRecord"].value_counts())

Jumlah data bersih per negara:


,count
country,
Singapore,761
Thailand,741
Viet Nam,728
Malaysia,720
Philippines,668
Indonesia,646


Jumlah data berdasarkan tahun:


,count
year,
2020,631
2021,635
2022,533
2023,639
2024,557
2025,644
2026,625


Jumlah data berdasarkan basisOfRecord:


,count
basisOfRecord,
HUMAN_OBSERVATION,4233
PRESERVED_SPECIMEN,31


In [ ]:
# save dataset
df_clean.to_csv(OUTPUT_FILE, index=False, encoding="utf-8-sig")
print(f"Dataset berhasil disimpan ke: {OUTPUT_FILE}")
display(df_clean.head())

Dataset berhasil disimpan ke: data/gbif_biodiversity_graph.csv


,occurrenceKey,taxonKey,scientificName,kingdomKey,kingdom,phylumKey,phylum,classKey,className,orderKey,...,speciesKey,species,countryCode,country,decimalLatitude,decimalLongitude,year,basisOfRecord,region,source
0,2542914614,2158332.0,"Poltys illepidus C.L.Koch, 1843",1.0,Animalia,54.0,Arthropoda,367.0,Arachnida,1496.0,...,2158332.0,Poltys illepidus,ID,Indonesia,-6.347636,106.759209,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
1,2542919124,3034128.0,Centella asiatica (L.) Urb.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1351.0,...,3034128.0,Centella asiatica,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
2,2542919768,3068937.0,Euphorbia hirta L.,6.0,Plantae,7707728.0,Tracheophyta,220.0,Magnoliopsida,1414.0,...,3068937.0,Euphorbia hirta,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
3,2543080789,2791514.0,Bryobium retusum (Blume) Y.P.Ng & P.J.Cribb,6.0,Plantae,7707728.0,Tracheophyta,196.0,Liliopsida,1169.0,...,2791514.0,Bryobium retusum,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API
4,2543081090,5307066.0,Crepidium koordersii (J.J.Sm.) Szlach.,6.0,Plantae,7707728.0,Tracheophyta,196.0,Liliopsida,1169.0,...,5307066.0,Crepidium koordersii,ID,Indonesia,-7.669757,110.201777,2020,HUMAN_OBSERVATION,Southeast Asia,GBIF Occurrence API


In [ ]:
# buat dataset teks untuk LLM Graph Builder
unique_species = (
    df_clean[["species", "genus", "family", "orderName", "className", "kingdom", "country", "region"]].drop_duplicates(subset=["species"]).head(50).copy())

def build_species_text(row):
    species = row["species"]
    genus = row["genus"]
    family = row["family"]
    order_name = row["orderName"]
    class_name = row["className"]
    kingdom = row["kingdom"]
    country = row["country"]
    region = row["region"]

    if kingdom == "Animalia":
        if class_name == "Aves":
            habitat = "wetlands, forests, gardens, agricultural areas, and urban environments"
            threat = "habitat degradation, pollution, and human disturbance"
        elif class_name == "Reptilia":
            habitat = "grasslands, shrubs, forests, and open terrestrial habitats"
            threat = "habitat loss, land use change, and human disturbance"
        elif class_name == "Mammalia":
            habitat = "forests, grasslands, and terrestrial ecosystems"
            threat = "habitat fragmentation, hunting, and human disturbance"
        elif class_name == "Insecta":
            habitat = "tropical forests, vegetation areas, gardens, and freshwater surroundings"
            threat = "habitat degradation, pesticide exposure, and environmental change"
        else:
            habitat = "natural habitats, forests, wetlands, and human-modified environments"
            threat = "habitat degradation and environmental change"
    elif kingdom == "Plantae":
        habitat = "tropical forests, open areas, disturbed habitats, and agricultural landscapes"
        threat = "land conversion, habitat degradation, and environmental change"
    else:
        habitat = "natural ecosystems and human-modified environments"
        threat = "environmental change and habitat degradation"
    return (
        f"{species} is a species from the genus {genus} and family {family}. "
        f"It belongs to the order {order_name}, class {class_name}, and kingdom {kingdom}. "
        f"This species has occurrence records in {country}, which is part of {region}. "
        f"It is commonly associated with habitats such as {habitat}. "
        f"Potential threats to this species include {threat}.")

unique_species["text"] = unique_species.apply(build_species_text, axis=1)
species_text_df = unique_species[["species", "text"]]
species_text_df.to_csv(SPECIES_TEXT_FILE, index=False, encoding="utf-8-sig")
print(f"Species description dataset berhasil disimpan ke: {SPECIES_TEXT_FILE}")
display(species_text_df.head())

Species description dataset berhasil disimpan ke: data/species_descriptions.csv


,species,text
0,Poltys illepidus,Poltys illepidus is a species from the genus P...
1,Centella asiatica,Centella asiatica is a species from the genus ...
2,Euphorbia hirta,Euphorbia hirta is a species from the genus Eu...
3,Bryobium retusum,Bryobium retusum is a species from the genus B...
4,Crepidium koordersii,Crepidium koordersii is a species from the gen...


In [ ]:
# preview semua file output
print("File output yang dibuat:")
print(f"1. Dataset utama          : {OUTPUT_FILE}")
print(f"2. Dataset graph builder  : {SPECIES_TEXT_FILE}")
print("\nKolom dataset utama:")
print(df_clean.columns.tolist())
print("\nUkuran dataset utama:")
print(df_clean.shape)
print("\nUkuran dataset graph builder:")
print(species_text_df.shape)

File output yang dibuat:
1. Dataset utama          : data/gbif_biodiversity_graph.csv
2. Dataset graph builder  : data/species_descriptions.csv

Kolom dataset utama:
['occurrenceKey', 'taxonKey', 'scientificName', 'kingdomKey', 'kingdom', 'phylumKey', 'phylum', 'classKey', 'className', 'orderKey', 'orderName', 'familyKey', 'family', 'genusKey', 'genus', 'speciesKey', 'species', 'countryCode', 'country', 'decimalLatitude', 'decimalLongitude', 'year', 'basisOfRecord', 'region', 'source']

Ukuran dataset utama:
(4264, 25)

Ukuran dataset graph builder:
(50, 2)


In [ ]:
from google.colab import files
files.download("data/gbif_biodiversity_graph.csv")
files.download("data/species_descriptions.csv")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>